# QRT Asset Allocation Performance Forecasting - LightGBM + CatBoost Baseline

**Objective:** Build a leakage-aware baseline for binary prediction of next-day return sign using grouped CV by timestamp (`TS`), compare LightGBM vs CatBoost, blend them, and export submission files.

> **Why grouped CV by `TS`?**
> Rows sharing the same timestamp can be highly related (cross-sectional market snapshot). Random row-level splits can leak timestamp-specific structure into validation. Grouped CV ensures each `TS` appears in only one fold, giving a more realistic estimate.

In [ ]:
# 1) Imports and global config
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import GroupKFold

from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5
np.random.seed(SEED)
random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

print("Environment ready. Seed:", SEED)

In [ ]:
# 2) Load data
DATA_DIR = Path(".")
required_files = ["X_train.csv", "y_train.csv", "X_test.csv", "sample_submission.csv"]

for fname in required_files:
    path = DATA_DIR / fname
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path.resolve()}")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Loaded:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("sample_submission:", sample_submission.shape)

In [ ]:
# 3) Inspect schema and basic sanity checks
print("\nX_train dtypes summary:")
print(X_train.dtypes.value_counts())

print("\nFirst columns:", X_train.columns[:15].tolist())

print("\nMissing values (train features):", int(X_train.isna().sum().sum()))
print("Missing values (test features) :", int(X_test.isna().sum().sum()))
print("Missing values (target file)   :", int(y_train.isna().sum().sum()))

In [ ]:
# 4) Merge X_train and y_train on ROW_ID, then build binary target
if "ROW_ID" not in X_train.columns or "ROW_ID" not in y_train.columns:
    raise KeyError("ROW_ID must be present in both X_train and y_train")

train_df = X_train.merge(y_train, on="ROW_ID", how="inner", validate="one_to_one")
assert len(train_df) == len(X_train), "Merged train length mismatch"

if "TARGET" not in train_df.columns:
    raise KeyError("TARGET column not found in y_train merge")

train_df["y_bin"] = (train_df["TARGET"] > 0).astype(int)

print("Merged train shape:", train_df.shape)
print("Binary target balance:")
print(train_df["y_bin"].value_counts(normalize=True).rename("proportion"))

In [ ]:
# 5) Column detection helpers and base preprocessing
id_candidates = ["ROW_ID", "TS", "ALLOCATION"]
id_cols = [c for c in id_candidates if c in train_df.columns]

if "ROW_ID" not in X_test.columns:
    raise KeyError("ROW_ID must exist in X_test for submission generation")

categorical_cols = [c for c in ["GROUP"] if c in train_df.columns]

ret_cols = [c for c in train_df.columns if c.startswith("RET_")]
vol_cols = [c for c in train_df.columns if c.startswith("SIGNED_VOLUME_")]

print("Detected ID columns:", id_cols)
print("Detected categorical columns:", categorical_cols)
print(f"Detected RET columns: {len(ret_cols)}")
print(f"Detected SIGNED_VOLUME columns: {len(vol_cols)}")

In [ ]:
# 6) Feature engineering

def _safe_divide(a, b, fill_value=0.0):
    b_safe = np.where(np.abs(b) < 1e-8, np.nan, b)
    out = a / b_safe
    out = np.where(np.isfinite(out), out, np.nan)
    return np.nan_to_num(out, nan=fill_value, posinf=fill_value, neginf=fill_value)


def add_rowwise_features(df):
    df = df.copy()

    if ret_cols:
        df["ret_mean_20"] = df[ret_cols].mean(axis=1)
        df["ret_std_20"] = df[ret_cols].std(axis=1)
        df["ret_min_20"] = df[ret_cols].min(axis=1)
        df["ret_max_20"] = df[ret_cols].max(axis=1)
        df["ret_pos_count_20"] = (df[ret_cols] > 0).sum(axis=1)

        abs_ret = df[ret_cols].abs()
        df["ret_abs_mean_20"] = abs_ret.mean(axis=1)
        df["ret_abs_std_20"] = abs_ret.std(axis=1)
        df["ret_abs_max_20"] = abs_ret.max(axis=1)

        for w in [3, 5, 10, 20]:
            cols = [f"RET_{i}" for i in range(1, w + 1) if f"RET_{i}" in df.columns]
            if cols:
                df[f"ret_mean_{w}"] = df[cols].mean(axis=1)

        if "RET_1" in df.columns and "ret_mean_5" in df.columns:
            df["ret_mom_1_minus_5"] = df["RET_1"] - df["ret_mean_5"]
        if "ret_mean_5" in df.columns and "ret_mean_20" in df.columns:
            df["ret_mom_5_minus_20"] = df["ret_mean_5"] - df["ret_mean_20"]

    if vol_cols:
        df["vol_mean_20"] = df[vol_cols].mean(axis=1)
        df["vol_std_20"] = df[vol_cols].std(axis=1)
        df["vol_min_20"] = df[vol_cols].min(axis=1)
        df["vol_max_20"] = df[vol_cols].max(axis=1)
        df["vol_pos_count_20"] = (df[vol_cols] > 0).sum(axis=1)

        for w in [3, 5, 10, 20]:
            cols = [f"SIGNED_VOLUME_{i}" for i in range(1, w + 1) if f"SIGNED_VOLUME_{i}" in df.columns]
            if cols:
                df[f"vol_mean_{w}"] = df[cols].mean(axis=1)

        if "SIGNED_VOLUME_1" in df.columns and "vol_mean_5" in df.columns:
            df["vol_trend_1_minus_5"] = df["SIGNED_VOLUME_1"] - df["vol_mean_5"]
        if "vol_mean_5" in df.columns and "vol_mean_20" in df.columns:
            df["vol_trend_5_minus_20"] = df["vol_mean_5"] - df["vol_mean_20"]

    if "MEDIAN_DAILY_TURNOVER" in df.columns:
        if "ret_mean_5" in df.columns:
            df["ret5_x_turnover"] = df["ret_mean_5"] * df["MEDIAN_DAILY_TURNOVER"]
        if "vol_mean_5" in df.columns:
            df["vol5_x_turnover"] = df["vol_mean_5"] * df["MEDIAN_DAILY_TURNOVER"]
        if "ret_abs_mean_20" in df.columns:
            df["absret20_to_turnover"] = _safe_divide(df["ret_abs_mean_20"].values, df["MEDIAN_DAILY_TURNOVER"].values)

    return df


def add_ts_cross_sectional_features(df, agg_cols):
    # Create TS-level cross-sectional aggregates within this dataframe only.
    df = df.copy()
    if "TS" not in df.columns:
        return df

    use_cols = [c for c in agg_cols if c in df.columns]
    if not use_cols:
        return df

    grouped = df.groupby("TS", observed=True)[use_cols].agg(["mean", "std"])
    grouped.columns = [f"ts_{col}_{stat}" for col, stat in grouped.columns]
    grouped = grouped.reset_index()

    df = df.merge(grouped, on="TS", how="left")
    return df


def build_features(df):
    out = add_rowwise_features(df)

    ts_agg_seed_cols = [
        "RET_1", "RET_5", "RET_10", "RET_20",
        "SIGNED_VOLUME_1", "SIGNED_VOLUME_5", "SIGNED_VOLUME_10", "SIGNED_VOLUME_20",
        "MEDIAN_DAILY_TURNOVER"
    ]
    out = add_ts_cross_sectional_features(out, ts_agg_seed_cols)

    if "GROUP" in out.columns:
        out["GROUP"] = out["GROUP"].astype("category")

    return out

In [ ]:
# 7) Prepare train/test engineered datasets
train_feat = build_features(train_df)
test_feat = build_features(X_test)

exclude_cols = set(id_cols + ["TARGET", "y_bin"])
feature_cols = [c for c in train_feat.columns if c not in exclude_cols]

missing_in_test = sorted(set(feature_cols) - set(test_feat.columns))
extra_in_test = sorted(set(test_feat.columns) - set(feature_cols) - set(id_cols))

if missing_in_test:
    raise ValueError(f"Test missing engineered columns: {missing_in_test[:10]}")
if extra_in_test:
    print(f"Info: {len(extra_in_test)} extra columns in test (ignored)")

X_all = train_feat[feature_cols].copy()
y_all = train_feat["y_bin"].copy()
groups = train_feat["TS"].copy() if "TS" in train_feat.columns else None

X_test_all = test_feat[feature_cols].copy()

for col in X_all.columns:
    if str(X_all[col].dtype) == "category":
        continue
    X_all[col] = pd.to_numeric(X_all[col], errors="coerce")
    X_test_all[col] = pd.to_numeric(X_test_all[col], errors="coerce")

X_all = X_all.replace([np.inf, -np.inf], np.nan)
X_test_all = X_test_all.replace([np.inf, -np.inf], np.nan)

for col in X_all.columns:
    if str(X_all[col].dtype) == "category":
        X_all[col] = X_all[col].cat.add_categories(["__MISSING__"]).fillna("__MISSING__")
        X_test_all[col] = X_test_all[col].astype("category").cat.add_categories(["__MISSING__"]).fillna("__MISSING__")
    else:
        # constant fill avoids fold-wise fitting leakage from validation data
        X_all[col] = X_all[col].fillna(0.0)
        X_test_all[col] = X_test_all[col].fillna(0.0)

print("Final train matrix:", X_all.shape)
print("Final test matrix :", X_test_all.shape)
print("Target mean:", y_all.mean())

In [ ]:
# 8) Evaluation helper

def evaluate_predictions(y_true, pred_proba, name="model"):
    pred_label = (pred_proba >= 0.5).astype(int)
    acc = accuracy_score(y_true, pred_label)
    ll = log_loss(y_true, np.clip(pred_proba, 1e-6, 1 - 1e-6))
    print(f"{name} -> Accuracy: {acc:.6f} | LogLoss: {ll:.6f}")
    return {"model": name, "oof_accuracy": acc, "oof_logloss": ll}

In [ ]:
# 9) Cross-validation runners

def run_lgbm_cv(X, y, X_test, groups, n_splits=5, seed=42):
    if groups is None:
        raise ValueError("TS column is required for grouped CV")

    splitter = GroupKFold(n_splits=n_splits)
    oof = np.zeros(len(X), dtype=float)
    test_preds = np.zeros(len(X_test), dtype=float)
    fold_accs, fold_lls, fi_list = [], [], []

    cat_cols = [c for c in X.columns if str(X[c].dtype) == "category"]

    for fold, (tr_idx, va_idx) in enumerate(splitter.split(X, y, groups=groups), start=1):
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = LGBMClassifier(
            objective="binary",
            n_estimators=2000,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_samples=80,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=seed + fold,
            n_jobs=-1,
        )

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="binary_logloss",
            categorical_feature=cat_cols if cat_cols else "auto",
            callbacks=[early_stopping(stopping_rounds=200), log_evaluation(200)],
        )

        va_prob = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = va_prob
        test_preds += model.predict_proba(X_test)[:, 1] / n_splits

        va_pred = (va_prob >= 0.5).astype(int)
        acc = accuracy_score(y_va, va_pred)
        ll = log_loss(y_va, np.clip(va_prob, 1e-6, 1 - 1e-6))
        fold_accs.append(acc)
        fold_lls.append(ll)

        fi_list.append(pd.Series(model.feature_importances_, index=X.columns, name=f"fold_{fold}"))
        print(f"[LGBM] Fold {fold}: acc={acc:.6f}, logloss={ll:.6f}")

    fi_df = pd.concat(fi_list, axis=1)
    fi_df["importance_mean"] = fi_df.mean(axis=1)
    fi_df = fi_df.sort_values("importance_mean", ascending=False)

    return {
        "oof": oof,
        "test_pred": test_preds,
        "fold_accs": fold_accs,
        "fold_lls": fold_lls,
        "feature_importance": fi_df,
    }


def run_catboost_cv(X, y, X_test, groups, n_splits=5, seed=42):
    if groups is None:
        raise ValueError("TS column is required for grouped CV")

    splitter = GroupKFold(n_splits=n_splits)
    oof = np.zeros(len(X), dtype=float)
    test_preds = np.zeros(len(X_test), dtype=float)
    fold_accs, fold_lls = [], []

    cat_cols = [c for c in X.columns if str(X[c].dtype) == "category"]

    for fold, (tr_idx, va_idx) in enumerate(splitter.split(X, y, groups=groups), start=1):
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        for c in cat_cols:
            X_tr[c] = X_tr[c].astype(str)
            X_va[c] = X_va[c].astype(str)

        X_test_fold = X_test.copy()
        for c in cat_cols:
            X_test_fold[c] = X_test_fold[c].astype(str)

        model = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="Accuracy",
            iterations=2500,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=8.0,
            random_seed=seed + fold,
            verbose=200,
        )

        model.fit(
            X_tr,
            y_tr,
            eval_set=(X_va, y_va),
            cat_features=cat_cols,
            use_best_model=True,
            early_stopping_rounds=200,
        )

        va_prob = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = va_prob
        test_preds += model.predict_proba(X_test_fold)[:, 1] / n_splits

        va_pred = (va_prob >= 0.5).astype(int)
        acc = accuracy_score(y_va, va_pred)
        ll = log_loss(y_va, np.clip(va_prob, 1e-6, 1 - 1e-6))
        fold_accs.append(acc)
        fold_lls.append(ll)
        print(f"[CAT] Fold {fold}: acc={acc:.6f}, logloss={ll:.6f}")

    return {
        "oof": oof,
        "test_pred": test_preds,
        "fold_accs": fold_accs,
        "fold_lls": fold_lls,
    }

In [ ]:
# 10) Run LightGBM CV
lgbm_results = run_lgbm_cv(X_all, y_all, X_test_all, groups=groups, n_splits=N_SPLITS, seed=SEED)
lgbm_eval = evaluate_predictions(y_all, lgbm_results["oof"], name="LightGBM OOF")

In [ ]:
# 11) Run CatBoost CV
cat_results = run_catboost_cv(X_all, y_all, X_test_all, groups=groups, n_splits=N_SPLITS, seed=SEED)
cat_eval = evaluate_predictions(y_all, cat_results["oof"], name="CatBoost OOF")

In [ ]:
# 12) Compare fold and OOF scores
comparison = pd.DataFrame([
    {
        "model": "LightGBM",
        **{f"fold_{i+1}_acc": v for i, v in enumerate(lgbm_results["fold_accs"])},
        "mean_fold_acc": np.mean(lgbm_results["fold_accs"]),
        "oof_acc": lgbm_eval["oof_accuracy"],
        "oof_logloss": lgbm_eval["oof_logloss"],
    },
    {
        "model": "CatBoost",
        **{f"fold_{i+1}_acc": v for i, v in enumerate(cat_results["fold_accs"])},
        "mean_fold_acc": np.mean(cat_results["fold_accs"]),
        "oof_acc": cat_eval["oof_accuracy"],
        "oof_logloss": cat_eval["oof_logloss"],
    },
])

comparison

In [ ]:
# 13) Blend probabilities (equal weights + small grid search)
lgbm_oof = lgbm_results["oof"]
cat_oof = cat_results["oof"]

equal_blend_oof = 0.5 * lgbm_oof + 0.5 * cat_oof
equal_metrics = evaluate_predictions(y_all, equal_blend_oof, name="Equal blend OOF")

weights = np.linspace(0.0, 1.0, 21)
blend_rows = []

for w in weights:
    blend_oof = w * lgbm_oof + (1 - w) * cat_oof
    acc = accuracy_score(y_all, (blend_oof >= 0.5).astype(int))
    ll = log_loss(y_all, np.clip(blend_oof, 1e-6, 1 - 1e-6))
    blend_rows.append((w, acc, ll))

blend_df = pd.DataFrame(blend_rows, columns=["w_lgbm", "oof_acc", "oof_logloss"]).sort_values(["oof_acc", "oof_logloss"], ascending=[False, True])
best_w = float(blend_df.iloc[0]["w_lgbm"])
print(f"Best blend weight on OOF (lgbm): {best_w:.2f}")
print(blend_df.head(10))

In [ ]:
# 14) Train final models on full training data
final_lgbm = LGBMClassifier(
    objective="binary",
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=80,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
)

cat_cols = [c for c in X_all.columns if str(X_all[c].dtype) == "category"]

final_lgbm.fit(X_all, y_all, categorical_feature=cat_cols if cat_cols else "auto")

X_all_cat = X_all.copy()
X_test_cat = X_test_all.copy()
for c in cat_cols:
    X_all_cat[c] = X_all_cat[c].astype(str)
    X_test_cat[c] = X_test_cat[c].astype(str)

final_cat = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="Accuracy",
    iterations=1800,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=8.0,
    random_seed=SEED,
    verbose=200,
)

final_cat.fit(X_all_cat, y_all, cat_features=cat_cols)

test_prob_lgbm = final_lgbm.predict_proba(X_test_all)[:, 1]
test_prob_cat = final_cat.predict_proba(X_test_cat)[:, 1]
test_prob_blend = best_w * test_prob_lgbm + (1 - best_w) * test_prob_cat

In [ ]:
# 15) Build submissions using sample_submission template
sub_template = sample_submission.copy()

id_like = {"ROW_ID", "row_id", "Id", "ID"}
pred_cols = [c for c in sub_template.columns if c not in id_like]
pred_col = pred_cols[0] if pred_cols else sub_template.columns[-1]

if "ROW_ID" in sub_template.columns and "ROW_ID" in X_test.columns:
    sub_template["ROW_ID"] = X_test["ROW_ID"].values

sub_lgbm = sub_template.copy()
sub_cat = sub_template.copy()
sub_blend = sub_template.copy()

sub_lgbm[pred_col] = (test_prob_lgbm >= 0.5).astype(int)
sub_cat[pred_col] = (test_prob_cat >= 0.5).astype(int)
sub_blend[pred_col] = (test_prob_blend >= 0.5).astype(int)

sub_lgbm.to_csv("submission_lgbm.csv", index=False)
sub_cat.to_csv("submission_catboost.csv", index=False)
sub_blend.to_csv("submission_blend.csv", index=False)

print("Saved: submission_lgbm.csv, submission_catboost.csv, submission_blend.csv")
print("Prediction column used:", pred_col)
sub_blend.head()

In [ ]:
# 16) Optional feature importance plot (LightGBM)
fi = lgbm_results["feature_importance"].head(20).sort_values("importance_mean", ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(fi.index, fi["importance_mean"])
plt.title("Top-20 LightGBM Feature Importances (mean across folds)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## Final summary

This notebook built a leakage-aware baseline pipeline for the QRT challenge:
- Merged train features and targets on `ROW_ID`, and converted `TARGET` into binary label `y_bin`.
- Engineered practical row-wise lag summary features, momentum/trend features, turnover interactions, and `TS` cross-sectional aggregates.
- Used **GroupKFold by `TS`** for robust validation.
- Trained and evaluated **LightGBM** and **CatBoost** with OOF probabilities.
- Compared folds, searched blend weights, and exported 3 submission files.

### Practical next steps
1. **Threshold tuning** on OOF probabilities (global and possibly by volatility regime).
2. **Probability calibration** (Platt / isotonic) to stabilize decisions.
3. Expand **date-level cross-sectional features** with robust stats (median/MAD, clipped moments).
4. Consider **pseudo-labeling** only if private/public leaderboard behavior supports it.
5. Add **diverse models** (e.g., regularized linear/logistic variants) for stronger ensembles.